# Assignment 6 — Base model baseline (before training)

Evaluate **`meta-llama/Llama-3.2-1B-Instruct`** on the ten graded questions **with no LoRA adapter**.

This notebook is for baseline comparison only. The graded deliverable after fine-tuning is `answers.ipynb`.

**Requirements:** GPU recommended (same local setup as Assignment 5), Hugging Face access to the Instruct checkpoint, and `HF_TOKEN` if the model is gated.

In [10]:
import os
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
MAX_NEW_TOKENS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Prefer an explicit env var; otherwise let huggingface_hub use the cached login token.
HF_TOKEN = os.environ.get("HF_TOKEN")
token_path = Path.home() / ".cache" / "huggingface" / "token"
if HF_TOKEN is None and token_path.is_file():
    HF_TOKEN = token_path.read_text(encoding="utf-8").strip() or None

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: running on CPU; generation will be slow.")

Model: meta-llama/Llama-3.2-1B-Instruct
Device: cuda
GPU: Quadro T2000


In [11]:
QUESTIONS = [
    (
        "If it takes 1 hour for 60 people to play an Opera, how many hours will "
        "it take 600 people to play the same opera?"
    ),
    "Is a pound of feathers or a British pound heavier?",
    (
        "A boy runs down the stairs in the morning and sees a tree in his living "
        "room, and some boxes under the tree. What's going on?"
    ),
    "What happens if you crack your knuckles a lot?",
    "If there is a shark in the pool of my basement, is it safe to go upstairs?",
    (
        "How much wood could a wood chuck chuck if there were only 5 pounds of "
        "wood in the world?"
    ),
    "Who is the current President of the United States?",
    "Was Talos alive?",
    "How many Ls are in the word parallel?",
    (
        "What is the riddle of the sphinx, and what are all possible answers "
        "satisfying all conditions?"
    ),
]

assert len(QUESTIONS) == 10

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if DEVICE == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    token=HF_TOKEN,
    attn_implementation="sdpa",
)
model.to(DEVICE)
model.eval()
print("Loaded base Instruct checkpoint (no adapter).")

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 712.70it/s]


Loaded base Instruct checkpoint (no adapter).


In [13]:
def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


baseline_answers = []
for index, question in enumerate(QUESTIONS, start=1):
    answer = ask(question)
    baseline_answers.append({"question": question, "answer": answer})
    print(f"=== Question {index} ===")
    print(question)
    print()
    print(answer)
    print()

=== Question 1 ===
If it takes 1 hour for 60 people to play an Opera, how many hours will it take 600 people to play the same opera?

To find the time it will take for 600 people to play the opera, we need to divide the time it takes for 60 people by 600.

Time for 60 people = 1 hour
Time for 600 people = 1 hour / 600

Time for 600 people = 1/600 hours

To convert this to a more meaningful unit, we can multiply by 60 (since there are 60 minutes in an hour):

Time for 600 people = (1/600) * 60
Time for 600 people = 1 minute

So, it will take 1 minute for

=== Question 2 ===
Is a pound of feathers or a British pound heavier?

A pound is a unit of weight, and it's a British unit of weight, but they are different things.

A pound is a unit of weight, typically used in the United States and some other countries. It's equal to 16 ounces or 453.592 grams.

A British pound, on the other hand, is a unit of currency, also known as a pound sterling. It's the official currency of the United Kingdo

In [14]:
import json
from datetime import UTC, datetime

out_dir = REPO_ROOT / "data" / "assignment6"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_before_training.json"
payload = {
    "model": MODEL_ID,
    "adapter": None,
    "device": DEVICE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": baseline_answers,
}
out_path.write_text(
    json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Wrote {out_path}")

Wrote /home/tshahzad/projects/USU-LLM-Class-F26/data/assignment6/baseline_before_training.json


## Score baseline vs gold

Compares model answers to `data/assignment6/gold_answers.json` using:

- **keywords** — must hit (any by default; all if `"match": "all"`)
- **reject_if** — automatic fail if any of these wrong phrases appear
- **soft metrics** — ROUGE + BERTScore against `expected`

Run the next cell → table + `data/assignment6/baseline_report.md`.


In [15]:
import json
import warnings
from pathlib import Path

from evaluate import load as load_metric

DATA = REPO_ROOT / "data" / "assignment6"
baseline = json.loads((DATA / "baseline_before_training.json").read_text())
gold = json.loads((DATA / "gold_answers.json").read_text())
assert len(baseline["examples"]) == len(gold) == 10


def norm(text: str) -> str:
    return " ".join(str(text).lower().split())


def has_phrase(text: str, phrase: str) -> bool:
    return norm(phrase) in norm(text)


def keyword_pass(prediction: str, item: dict) -> bool:
    if any(has_phrase(prediction, bad) for bad in item.get("reject_if") or []):
        return False
    keywords = item.get("keywords") or []
    if not keywords:
        return False
    hits = [kw for kw in keywords if has_phrase(prediction, kw)]
    if item.get("match") == "all":
        return len(hits) == len(keywords)
    return bool(hits)


rows = []
for pred, item in zip(baseline["examples"], gold, strict=True):
    rows.append(
        {
            "id": item["id"],
            "pass": keyword_pass(pred["answer"], item),
            "expected": item["expected"],
            "model": pred["answer"],
            "question": item["question"],
        }
    )

preds = [r["model"] for r in rows]
refs = [r["expected"] for r in rows]

# Quiet the noisy Roberta load report from BERTScore.
warnings.filterwarnings("ignore")
import logging

import transformers

transformers.logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

rouge = load_metric("rouge").compute(
    predictions=preds,
    references=refs,
    rouge_types=["rouge1", "rougeL"],
    use_stemmer=False,
)
bert = load_metric("bertscore").compute(
    predictions=preds,
    references=refs,
    lang="en",
    model_type="roberta-base",
    idf=False,
    rescale_with_baseline=False,
    device="cpu",
)
soft = {
    "rouge1": round(float(rouge["rouge1"]), 3),
    "rougeL": round(float(rouge["rougeL"]), 3),
    "bertscore_f1": round(float(sum(bert["f1"]) / len(bert["f1"])), 3),
}
pass_rate = sum(r["pass"] for r in rows) / len(rows)

print(f"{'ID':>2}  {'PASS':4}  QUESTION")
print("-" * 72)
for r in rows:
    print(f"{r['id']:>2}  {r['pass']!s:4}  {r['question'][:60]}")

print()
print(f"Keyword pass rate      : {pass_rate:.0%}")
print(f"Soft ROUGE-1 / ROUGE-L : {soft['rouge1']} / {soft['rougeL']}")
print(f"Soft BERTScore F1      : {soft['bertscore_f1']}")

lines = [
    "# Baseline score report",
    "",
    f"- Model: `{baseline.get('model')}`",
    f"- Keyword pass: **{pass_rate:.0%}** ({sum(r['pass'] for r in rows)}/10)",
    f"- Soft: ROUGE-1={soft['rouge1']}, ROUGE-L={soft['rougeL']}, BERTScore F1={soft['bertscore_f1']}",
    "",
    "| ID | Pass | Expected |",
    "| ---: | :---: | --- |",
]
for r in rows:
    lines.append(
        f"| {r['id']} | {'✓' if r['pass'] else '✗'} | {r['expected'].replace('|', '/')} |"
    )

lines += ["", "## Per-question detail", ""]
for r in rows:
    lines += [
        f"### Q{r['id']}. {r['question']}",
        "",
        f"**Expected:** {r['expected']}",
        "",
        f"**Model:** {r['model']}",
        "",
        f"- Keywords: {'pass' if r['pass'] else 'fail'}",
        "",
    ]

report = DATA / "baseline_report.md"
report.write_text("\n".join(lines), encoding="utf-8")
print(f"\nWrote {report}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 15207.10it/s]


ID  PASS  QUESTION
------------------------------------------------------------------------
 1  False  If it takes 1 hour for 60 people to play an Opera, how many 
 2  False  Is a pound of feathers or a British pound heavier?
 3  False  A boy runs down the stairs in the morning and sees a tree in
 4  False  What happens if you crack your knuckles a lot?
 5  False  If there is a shark in the pool of my basement, is it safe t
 6  False  How much wood could a wood chuck chuck if there were only 5 
 7  False  Who is the current President of the United States?
 8  False  Was Talos alive?
 9  False  How many Ls are in the word parallel?
10  False  What is the riddle of the sphinx, and what are all possible 

Keyword pass rate      : 0%
Soft ROUGE-1 / ROUGE-L : 0.097 / 0.077
Soft BERTScore F1      : 0.824

Wrote /home/tshahzad/projects/USU-LLM-Class-F26/data/assignment6/baseline_report.md
